# Chapter 19: Root Finding

**MAD 2502 — Computational Mathematics with Python**

A **root** (or zero) of a function is a value $x$ such that $f(x)=0$. Numerical root-finding methods approximate roots when an exact algebraic solution is unavailable or inconvenient.

**Learning rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect.

## Learning goals

You should be able to identify a root graphically, implement bisection, explain Newton's method, use a stopping tolerance and maximum iteration count, recognize important failure cases, and verify results with SciPy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import fsolve

## Understand: a root is where $f(x)=0$

For $f(x)=x^2-2$, the real roots are $\pm\sqrt{2}$. A graph can suggest where roots are, but a numerical method gives a systematic approximation.

In [ ]:
def f(x):
    return x**2 - 2


x = np.linspace(-2, 2, 400)
fig, ax = plt.subplots()
ax.plot(x, f(x))
ax.axhline(0, linestyle="--")
ax.set(xlabel="x", ylabel="f(x)", title="Roots occur where the curve crosses y=0")
plt.show()

## Bisection

If a continuous function changes sign between `a` and `b`, then at least one root lies in the interval. Bisection repeatedly cuts the interval in half and keeps the half containing a sign change.

In [ ]:
def bisection(function, a, b, tolerance=1e-10, max_iterations=100):
    fa = function(a)
    fb = function(b)
    if fa == 0:
        return a
    if fb == 0:
        return b
    if fa * fb > 0:
        raise ValueError("The endpoints must bracket a sign change.")

    for _ in range(max_iterations):
        midpoint = (a + b) / 2
        fm = function(midpoint)
        if abs(fm) < tolerance or (b - a) / 2 < tolerance:
            return midpoint
        if fa * fm < 0:
            b = midpoint
            fb = fm
        else:
            a = midpoint
            fa = fm

    raise RuntimeError("Bisection did not converge within max_iterations.")


root = bisection(f, 1, 2)
print(root, f(root))

## Predict → Run

Start with `[1, 2]`. The first midpoint is 1.5. Since $f(1)<0$ and $f(1.5)>0$, predict which half interval bisection keeps next.

## Newton's method

Newton's method uses the tangent-line approximation

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

It can converge very quickly near a suitable root, but unlike bisection it does not automatically stay inside a safe bracket.

In [ ]:
def newton(function, derivative, x0, tolerance=1e-10, max_iterations=50):
    x = float(x0)
    for _ in range(max_iterations):
        fx = function(x)
        if abs(fx) < tolerance:
            return x
        dfx = derivative(x)
        if abs(dfx) < 1e-14:
            raise ZeroDivisionError("Derivative is too close to zero.")
        x = x - fx / dfx
    raise RuntimeError("Newton method did not converge within max_iterations.")


def df(x):
    return 2 * x


root = newton(f, df, 1.0)
print(root, f(root))

## Change

Try Newton's method from `x0=10`, then `x0=-1`. Compare the roots reached and the number of conceptual steps needed. Starting values matter.

## A classic Newton failure: cube root of $x$

Let $f(x)=\sqrt[3]{x}$. For any nonzero starting value, Newton's update becomes

$$x_{n+1}=x_n-\frac{x_n^{1/3}}{(1/3)x_n^{-2/3}}=-2x_n.$$

So the iterates double in magnitude and alternate sign: Newton's method diverges for every nonzero starting point in this example.

In [ ]:
x_value = 1.0
for step in range(8):
    print(step, x_value)
    x_value = -2 * x_value

This failure is not a Python bug. It follows directly from the Newton update. A method's assumptions and geometry matter.

## Multiple roots and initial guesses

Consider

$$p(x)=x^3-100x^2-x+100=(x-100)(x-1)(x+1).$$

It has **three** real roots: `-1`, `1`, and `100`. A root finder may return different roots depending on the initial guess or bracket.

In [ ]:
def p(x):
    return x**3 - 100 * x**2 - x + 100


for guess in [-2, 0.5, 50, 120]:
    found = fsolve(p, guess)[0]
    print("guess:", guess, "root:", found)

## Method comparison

| Method | Main requirement | Strength | Common problem |
|---|---|---|---|
| Bisection | continuous function + sign-changing bracket | reliable and predictable | slower |
| Newton | function + derivative + useful starting guess | often very fast | can diverge or hit small derivative |
| `fsolve` | reasonable initial guess | convenient library solver | result depends on problem/guess; still verify |

Always check the residual `abs(f(root))` rather than trusting a returned number blindly.

## Practice

1. Use bisection to solve $x^3-x-2=0$ on `[1, 2]`.
2. Derive the Newton update for $f(x)=x^2-5$.
3. Compare bisection and Newton for the same root.
4. Find an example where a poor Newton starting point causes trouble.
5. For every approximate root, report `f(root)` as a verification.

## Connect

Root finding is a model for numerical mathematics: define the problem, choose a method whose assumptions fit, set a tolerance, limit iterations, test failure cases, and verify the answer. Those habits carry directly into optimization, simulation, regression, and capstone work.